# Sesión 07 — Diseño de red de acceso sobre San Isidro

Notebook de diseño construido **fase por fase** siguiendo el flujo
profesional (ver `index.md` y `brainstorming-diseno-red.md`).
El esbozo exploratorio previo vive en `test_scene.ipynb`.

Kernel: `ran-design` (sionna 2.x). Ejecución headless:
`python -m nbconvert --to notebook --execute --inplace design.ipynb`

## Fase 0 — Requisitos del encargo

Todo el diseño se verifica contra estas metas (tabla completa y
justificación en `index.md` §Fase 0). Son constantes del proyecto:
si el negocio las cambia, el diseño se recalcula — por eso viven en
una celda propia al inicio.

In [1]:
# ============ REQUISITOS (Fase 0) — contrato del diseño ============
REQ = {
    # R1 — área de servicio: la escena de San Isidro
    "escena":            "blends/test_scene/untitled.xml",
    "area_km2":          1.32 * 0.83,
    # R2 — cobertura de control
    "rsrp_min_dbm":      -110.0,
    "rsrp_prob":         0.95,
    # R3 — calidad de datos
    "sinr_min_db":       0.0,
    "sinr_prob":         0.90,
    # R4 — throughput de borde (percentil 5)
    "thr_borde_dl_mbps": 50.0,
    "thr_borde_ul_mbps": 5.0,
    # R5 — capacidad agregada en hora cargada
    "capacidad_mbps_km2": 600.0,
    # R7 — espectro licenciado
    "banda":             "n78",
    "fc_hz":             3.5e9,
    "bw_hz":             100e6,
    # R8 — restricciones de despliegue
    "max_sitios":        6,
    "p_tx_dbm_max":      44.0,
}

# La cuenta detrás de R5 (hipótesis de negocio explícitas):
personas_km2, market_share = 25_000, 0.30
gb_mes, f_bh = 10, 0.10
kbps_por_abonado = gb_mes * 8e9 * f_bh / (30 * 3600) / 1e3
demanda = personas_km2 * market_share * kbps_por_abonado / 1e3   # Mbps/km2
print(f"{kbps_por_abonado:.0f} kbps/abonado en hora cargada "
      f"-> demanda {demanda:.0f} Mbps/km2 (requisito: {REQ['capacidad_mbps_km2']:.0f})")
assert demanda <= REQ["capacidad_mbps_km2"], "R5 no cubre la demanda estimada"

74 kbps/abonado en hora cargada -> demanda 556 Mbps/km2 (requisito: 600)


## Fase 1 — Estrategia de espectro

Dos cuentas que fijan el resto del diseño (teoría en `index.md` §Fase 1):

1. **Frecuencia → sitios**: el delta de pérdida entre bandas se convierte en
   factor de radio ($10^{\Delta/10n}$) y de área — la razón física de las
   capas de espectro.
2. **TDD → ancho de banda efectivo**: el patrón DDDSU reparte el tiempo;
   DL y UL no ven los mismos 100 MHz.

In [2]:
import numpy as np

# ---- Cuenta 1: bandas candidatas vs n78 (referencia del encargo) ----
N_PROP = 3.8                       # exponente de propagación urbano denso
bandas_hz = {"n28 (700 MHz)": 0.7e9, "n1 (2.1 GHz)": 2.1e9,
             "n78 (3.5 GHz)": 3.5e9, "n258 (26 GHz)": 26e9}

fc_ref = REQ["fc_hz"]
print(f"{'banda':<15} {'Δpérdida':>9} {'radio rel.':>10} {'sitios rel.':>11}")
for nombre, f in bandas_hz.items():
    delta_db = 20*np.log10(f/fc_ref)          # solo el término de frecuencia
    r_rel = 10**(-delta_db/(10*N_PROP))       # MAPL fijo -> radio relativo
    sitios_rel = 1/r_rel**2                   # sitios ∝ 1/área de celda
    print(f"{nombre:<15} {delta_db:>+7.1f}dB {r_rel:>9.2f}x {sitios_rel:>10.2f}x")

print("\nLección: la banda se paga en sitios — o al revés.")

banda            Δpérdida radio rel. sitios rel.
n28 (700 MHz)     -14.0dB      2.33x       0.18x
n1 (2.1 GHz)       -4.4dB      1.31x       0.58x
n78 (3.5 GHz)      +0.0dB      1.00x       1.00x
n258 (26 GHz)     +17.4dB      0.35x       8.26x

Lección: la banda se paga en sitios — o al revés.


In [3]:
# ---- Cuenta 2: patrón TDD DDDSU -> ancho de banda efectivo ----
# De cada 5 slots: 3 DL + 1 especial (~57% de símbolos DL) + 1 UL
slots = {"DL": 3, "S": 1, "UL": 1}
frac_dl = (slots["DL"] + 0.57*slots["S"]) / sum(slots.values())
frac_ul = slots["UL"] / sum(slots.values())

ESPECTRO = {                       # decisión de la Fase 1 — hereda el diseño
    "patron_tdd":   "DDDSU",
    "frac_dl":      frac_dl,
    "frac_ul":      frac_ul,
    "bw_dl_ef_hz":  REQ["bw_hz"] * frac_dl,
    "bw_ul_ef_hz":  REQ["bw_hz"] * frac_ul,
    "scs_hz":       30e3,          # numerologia mu=1 (S03); CP 2.3 us >> DS 60 ns medido
}
print(f"DL: {frac_dl:.0%} del tiempo -> {ESPECTRO['bw_dl_ef_hz']/1e6:.0f} MHz efectivos")
print(f"UL: {frac_ul:.0%} del tiempo -> {ESPECTRO['bw_ul_ef_hz']/1e6:.0f} MHz efectivos")
print("El UE pierde dos veces: en potencia (23 vs 44 dBm) y en tiempo (1 slot de 5).")

DL: 71% del tiempo -> 71 MHz efectivos
UL: 20% del tiempo -> 20 MHz efectivos
El UE pierde dos veces: en potencia (23 vs 44 dBm) y en tiempo (1 slot de 5).


## Fase 2 — Dimensionamiento por cobertura

Link budget → MAPL → radio (UMa 3GPP) → sitios por cobertura.
Teoría en `index.md` §Fase 2; la mecánica per-RE en `rsrp-rsrq-sinr.md`.

Regla de la fase: el radio real lo fija el **peor** de los dos enlaces
(DL de control vs UL de datos).

In [4]:
# ---- Link budget DL (cobertura de control, R2) ----
N_PRB = 273                                          # 100 MHz @ SCS 30 kHz (TS 38.101)
n_sc = N_PRB * 12                                    # subportadoras utilizables
EPRE = REQ["p_tx_dbm_max"] - 10*np.log10(n_sc)       # dBm por RE: el 44 se reparte
G_TX, M_SHADOW = 16.0, 9.0                           # dBi macro / margen 95%, sigma 8 dB

mapl_dl = EPRE + G_TX + (-REQ["rsrp_min_dbm"]) - M_SHADOW
print(f"EPRE = {EPRE:.1f} dBm  (los 44 dBm entre {n_sc} subportadoras)")
print(f"MAPL DL (control) = {mapl_dl:.1f} dB")

# ---- Link budget UL (datos, R4-UL) ----
# El UE concentra 23 dBm en pocos PRB asignados (aqui 5 MHz) — su gran defensa
P_UE, NF_BS, SNR_UL_MIN, BW_UL = 23.0, 5.0, 0.0, 5e6
sens_bs = -174 + 10*np.log10(BW_UL) + NF_BS + SNR_UL_MIN
mapl_ul = P_UE + G_TX - sens_bs - M_SHADOW           # misma antena BS recibe
print(f"sensibilidad BS = {sens_bs:.1f} dBm  ->  MAPL UL = {mapl_ul:.1f} dB")

MAPL = min(mapl_dl, mapl_ul)
limitante = "DL control" if mapl_dl < mapl_ul else "UL datos"
print(f"\nMAPL de diseño = {MAPL:.1f} dB  (enlace limitante: {limitante})")

EPRE = 8.8 dBm  (los 44 dBm entre 3276 subportadoras)
MAPL DL (control) = 125.8 dB
sensibilidad BS = -102.0 dBm  ->  MAPL UL = 132.0 dB

MAPL de diseño = 125.8 dB  (enlace limitante: DL control)


In [5]:
# ---- MAPL -> radio (UMa NLOS, 3GPP TR 38.901) -> sitios por cobertura ----
fc_ghz = REQ["fc_hz"]/1e9

def radio_uma_nlos(mapl_db):
    # PL = 13.54 + 39.08 log10(d) + 20 log10(fc)   [d en m, fc en GHz]
    return 10**((mapl_db - 13.54 - 20*np.log10(fc_ghz)) / 39.08)

r_m = radio_uma_nlos(MAPL)
area_sitio_km2 = 2.6 * (r_m/1e3)**2                 # hexagono trisectorial
n_cobertura = int(np.ceil(REQ["area_km2"] / area_sitio_km2))

print(f"radio de celda      = {r_m:.0f} m")
print(f"area por sitio      = {area_sitio_km2:.2f} km2")
print(f"sitios por COBERTURA = {n_cobertura}  (presupuesto R8: {REQ['max_sitios']})")
assert n_cobertura <= REQ["max_sitios"], "la cobertura sola ya excede R8"

COBERTURA = {"mapl_db": MAPL, "limitante": limitante,
             "radio_m": r_m, "n_sitios": n_cobertura}
# Nota: la Fase 3 hara la cuenta gemela por capacidad; el diseno final
# necesita max(cobertura, capacidad). La validacion calle-a-calle contra
# la escena real de San Isidro llega en la Fase 6.

radio de celda      = 394 m
area por sitio      = 0.40 km2
sitios por COBERTURA = 3  (presupuesto R8: 6)


## Fase 3 — Dimensionamiento por capacidad

La cuenta gemela de la Fase 2: sitios para que la red **aguante** R5.
Concepto difícil (qué media usa el scheduler): nota
`de-sinr-a-capacidad.md`. Veredicto de diseño: `max(cobertura, capacidad)`.

In [6]:
# ---- Capacidad de celda y de sitio (DL) ----
SE_CELDA = 2.0      # bit/s/Hz — HIPOTESIS industria SISO conservadora.
                    # ponytail: la Fase 6 la reemplaza con la integral del
                    # mapa SINR ray-traced; queda declarada, no escondida.
OH = 0.22           # overhead: SSB + PDCCH + DMRS (PRBs que no venden bits)

r_celda = SE_CELDA * ESPECTRO["bw_dl_ef_hz"] * (1 - OH)          # bit/s
r_sitio = 3 * r_celda                                            # 3 sectores
print(f"celda: {r_celda/1e6:.0f} Mbps | sitio (3 sectores): {r_sitio/1e6:.0f} Mbps")

# ---- Demanda total vs sitios ----
demanda_total = REQ["capacidad_mbps_km2"] * REQ["area_km2"] * 1e6  # bit/s
n_capacidad = int(np.ceil(demanda_total / r_sitio))

N_SITIOS = max(COBERTURA["n_sitios"], n_capacidad)
print(f"demanda R5: {demanda_total/1e6:.0f} Mbps")
print(f"sitios por CAPACIDAD = {n_capacidad}")
print(f"\nVEREDICTO: max(cobertura={COBERTURA['n_sitios']}, "
      f"capacidad={n_capacidad}) = {N_SITIOS} sitios")
assert N_SITIOS <= REQ["max_sitios"], "el diseño excede el presupuesto R8"

# ---- Margen de crecimiento: ¿cuándo manda la capacidad? ----
instalada = COBERTURA["n_sitios"] * r_sitio
margen = instalada / demanda_total
anios = np.log(margen) / np.log(1.25)          # trafico +25%/año
print(f"capacidad instalada {instalada/1e6:.0f} Mbps -> margen {margen:.2f}x "
      f"≈ {anios:.1f} años al 25%/año")

CAPACIDAD = {"se_celda": SE_CELDA, "overhead": OH, "r_sitio_bps": r_sitio,
             "n_sitios": n_capacidad, "n_final": N_SITIOS}

celda: 111 Mbps | sitio (3 sectores): 334 Mbps
demanda R5: 657 Mbps
sitios por CAPACIDAD = 2

VEREDICTO: max(cobertura=3, capacidad=2) = 3 sitios
capacidad instalada 1002 Mbps -> margen 1.52x ≈ 1.9 años al 25%/año


## Fase 4 — Plan nominal *(pendiente)*

## Fase 5 — Planificación detallada *(pendiente)*

## Fase 6 — Validación *(pendiente)*